# Synthetic Data Generation Using RAGAS - RAG Evaluation with LangSmith

In the following notebook we'll explore a use-case for RAGAS' synthetic testset generation workflow!



- 🤝 BREAKOUT ROOM #1
  1. Use RAGAS to Generate Synthetic Data

- 🤝 BREAKOUT ROOM #2
  1. Load them into a LangSmith Dataset
  2. Evaluate our RAG chain against the synthetic test data
  3. Make changes to our pipeline
  4. Evaluate the modified pipeline

SDG is a critical piece of the puzzle, especially for early iteration! Without it, it would not be nearly as easy to get high quality early signal for our application's performance.

Let's dive in!

# 🤝 BREAKOUT ROOM #1

## Task 1: Dependencies and API Keys

We'll need to install a number of API keys and dependencies, since we'll be leveraging a number of great technologies for this pipeline!

1. OpenAI's endpoints to handle the Synthetic Data Generation
2. OpenAI's Endpoints for our RAG pipeline and LangSmith evaluation
3. QDrant as our vectorstore
4. LangSmith for our evaluation coordinator!

Let's install and provide all the required information below!

## Dependencies and API Keys:

### NLTK Import

To prevent errors that may occur based on OS - we'll import NLTK and download the needed packages to ensure correct handling of data.

In [1]:
import sys
print(f"Python executable: {sys.executable}")
print(f"Python version: {sys.version}")

Python executable: c:\Work\AIE_Course\AIE8\07_Synthetic_Data_Generation_and_LangSmith\.venv\Scripts\python.exe
Python version: 3.13.7 (tags/v3.13.7:bcee1c3, Aug 14 2025, 14:15:11) [MSC v.1944 64 bit (AMD64)]


In [2]:
import nltk
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')

C:\Users\brank\AppData\Local\Programs\Python\Python313\Lib\ssl.py:524: UserWarning: Bad certificate in Windows certificate store: not enough data: cadata does not contain a certificate (_ssl.c:4219)
  warnings.warn(f"Bad certificate in Windows certificate store: {exc!s}")
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\brank\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\brank\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!


True

In [3]:
import os
import getpass

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = getpass.getpass("LangChain API Key:")

We'll also want to set a project name to make things easier for ourselves.

In [4]:
from uuid import uuid4

os.environ["LANGCHAIN_PROJECT"] = f"AIM - SDG - {uuid4().hex[0:8]}"

OpenAI's API Key!

In [5]:
os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API Key:")

## Generating Synthetic Test Data

We wil be using Ragas to build out a set of synthetic test questions, references, and reference contexts. This is useful because it will allow us to find out how our system is performing.

> NOTE: Ragas is best suited for finding *directional* changes in your LLM-based systems. The absolute scores aren't comparable in a vacuum.

### Data Preparation

We'll prepare our data - which should hopefull be familiar at this point since it's our Use-Case Data!

Next, let's load our data into a familiar LangChain format using the `DirectoryLoader`.

In [6]:
from langchain_community.document_loaders import DirectoryLoader
from langchain_community.document_loaders import PyMuPDFLoader


path = "data/"
loader = DirectoryLoader(path, glob="*.pdf", loader_cls=PyMuPDFLoader)
docs = loader.load()

### Knowledge Graph Based Synthetic Generation

Ragas uses a knowledge graph based approach to create data. This is extremely useful as it allows us to create complex queries rather simply. The additional testset complexity allows us to evaluate larger problems more effectively, as systems tend to be very strong on simple evaluation tasks.

Let's start by defining our `generator_llm` (which will generate our questions, summaries, and more), and our `generator_embeddings` which will be useful in building our graph.

### Unrolled SDG

In [7]:
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-nano"))
generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings())

C:\Users\brank\AppData\Local\Programs\Python\Python313\Lib\ssl.py:524: UserWarning: Bad certificate in Windows certificate store: not enough data: cadata does not contain a certificate (_ssl.c:4219)
  warnings.warn(f"Bad certificate in Windows certificate store: {exc!s}")


Next, we're going to instantiate our Knowledge Graph.

This graph will contain N number of nodes that have M number of relationships. These nodes and relationships (AKA "edges") will define our knowledge graph and be used later to construct relevant questions and responses.

In [8]:
from ragas.testset.graph import KnowledgeGraph

kg = KnowledgeGraph()
kg

KnowledgeGraph(nodes: 0, relationships: 0)

The first step we're going to take is to simply insert each of our full documents into the graph. This will provide a base that we can apply transformations to.

In [9]:
from ragas.testset.graph import Node, NodeType

### NOTICE: We're using a subset of the data for this example - this is to keep costs/time down.
for doc in docs:
    kg.nodes.append(
        Node(
            type=NodeType.DOCUMENT,
            properties={"page_content": doc.page_content, "document_metadata": doc.metadata}
        )
    )
kg

KnowledgeGraph(nodes: 64, relationships: 0)

Now, we'll apply the *default* transformations to our knowledge graph. This will take the nodes currently on the graph and transform them based on a set of [default transformations](https://docs.ragas.io/en/latest/references/transforms/#ragas.testset.transforms.default_transforms).

These default transformations are dependent on the corpus length, in our case:

- Producing Summaries -> produces summaries of the documents
- Extracting Headlines -> finding the overall headline for the document
- Theme Extractor -> extracts broad themes about the documents

It then uses cosine-similarity and heuristics between the embeddings of the above transformations to construct relationships between the nodes.

In [10]:
from ragas.testset.transforms import default_transforms, apply_transforms

transformer_llm = generator_llm
embedding_model = generator_embeddings

default_transforms = default_transforms(documents=docs, llm=transformer_llm, embedding_model=embedding_model)
apply_transforms(kg, default_transforms)
kg

Applying HeadlinesExtractor:   0%|          | 0/21 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/64 [00:00<?, ?it/s]

unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to ap

Applying SummaryExtractor:   0%|          | 0/38 [00:00<?, ?it/s]

Property 'summary' already exists in node '57d319'. Skipping!
Property 'summary' already exists in node 'd3a806'. Skipping!
Property 'summary' already exists in node '46b7ec'. Skipping!
Property 'summary' already exists in node '36fe51'. Skipping!
Property 'summary' already exists in node 'e7ee9d'. Skipping!
Property 'summary' already exists in node 'babad6'. Skipping!
Property 'summary' already exists in node '9c8ce8'. Skipping!
Property 'summary' already exists in node 'cddef4'. Skipping!
Property 'summary' already exists in node 'cddc18'. Skipping!
Property 'summary' already exists in node 'b4af6e'. Skipping!
Property 'summary' already exists in node '9b27fc'. Skipping!
Property 'summary' already exists in node 'f96f56'. Skipping!
Property 'summary' already exists in node 'f54141'. Skipping!
Property 'summary' already exists in node '96594a'. Skipping!
Property 'summary' already exists in node '8e5957'. Skipping!
Property 'summary' already exists in node '105146'. Skipping!
Property

Applying CustomNodeFilter:   0%|          | 0/8 [00:00<?, ?it/s]

Applying [EmbeddingExtractor, ThemesExtractor, NERExtractor]:   0%|          | 0/48 [00:00<?, ?it/s]

Property 'summary_embedding' already exists in node 'cddef4'. Skipping!
Property 'summary_embedding' already exists in node 'cddc18'. Skipping!
Property 'summary_embedding' already exists in node 'd3a806'. Skipping!
Property 'summary_embedding' already exists in node '36fe51'. Skipping!
Property 'summary_embedding' already exists in node 'babad6'. Skipping!
Property 'summary_embedding' already exists in node '57d319'. Skipping!
Property 'summary_embedding' already exists in node '9b27fc'. Skipping!
Property 'summary_embedding' already exists in node 'e7ee9d'. Skipping!
Property 'summary_embedding' already exists in node '9c8ce8'. Skipping!
Property 'summary_embedding' already exists in node 'b4af6e'. Skipping!
Property 'summary_embedding' already exists in node '46b7ec'. Skipping!
Property 'summary_embedding' already exists in node 'f54141'. Skipping!
Property 'summary_embedding' already exists in node 'f96f56'. Skipping!
Property 'summary_embedding' already exists in node '8e5957'. Sk

Applying [CosineSimilarityBuilder, OverlapScoreBuilder]:   0%|          | 0/2 [00:00<?, ?it/s]

KnowledgeGraph(nodes: 86, relationships: 712)

In [11]:
import json
from ragas.testset.graph import KnowledgeGraph, UUIDEncoder, Node, Relationship

# Monkey patch the save method to use UTF-8
def save_utf8(self, path):
    data = {
        "nodes": [node.model_dump() for node in self.nodes],
        "relationships": [rel.model_dump() for rel in self.relationships],
    }
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, cls=UUIDEncoder, indent=2, ensure_ascii=False)

# Monkey patch the load method to use UTF-8
@classmethod
def load_utf8(cls, path):
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    nodes = [Node(**node) for node in data["nodes"]]
    relationships = [Relationship(**rel) for rel in data["relationships"]]
    kg = cls()
    kg.nodes = nodes
    kg.relationships = relationships
    return kg

KnowledgeGraph.save = save_utf8
KnowledgeGraph.load = load_utf8

# Now save and load
kg.save("usecase_data_kg.json")
usecase_data_kg = KnowledgeGraph.load("usecase_data_kg.json")
usecase_data_kg

KnowledgeGraph(nodes: 86, relationships: 712)

Using our knowledge graph, we can construct a "test set generator" - which will allow us to create queries.

In [12]:
from ragas.testset import TestsetGenerator

generator = TestsetGenerator(llm=generator_llm, embedding_model=embedding_model, knowledge_graph=usecase_data_kg)

However, we'd like to be able to define the kinds of queries we're generating - which is made simple by Ragas having pre-created a number of different "QuerySynthesizer"s.

Each of these Synthetsizers is going to tackle a separate kind of query which will be generated from a scenario and a persona.

In essence, Ragas will use an LLM to generate a persona of someone who would interact with the data - and then use a scenario to construct a question from that data and persona.

In [13]:
from ragas.testset.synthesizers import default_query_distribution, SingleHopSpecificQuerySynthesizer, MultiHopAbstractQuerySynthesizer, MultiHopSpecificQuerySynthesizer

query_distribution = [
        (SingleHopSpecificQuerySynthesizer(llm=generator_llm), 0.5),
        (MultiHopAbstractQuerySynthesizer(llm=generator_llm), 0.25),
        (MultiHopSpecificQuerySynthesizer(llm=generator_llm), 0.25),
]

#### ❓ Question #1:

What are the three types of query synthesizers doing? Describe each one in simple terms.

##### ✅ Answer:

1. SingleHopSpecificQuerySynthesizer: Creates simple, straightforward questions that can be answered using information from a single document or chunk. These are direct questions with specific answers.
Example: "What is the launch date of ChatGPT?"
2. MultiHopAbstractQuerySynthesizer: Generates complex questions that require combining information from multiple sources AND need higher-level reasoning or summarization. These questions ask for broader insights or patterns.
Example: "How do ChatGPT's usage patterns relate to productivity improvements across different occupations?"
3. MultiHopSpecificQuerySynthesizer: Creates questions that need information from multiple documents/chunks but ask for specific factual answers rather than abstract reasoning.
Example: "How many users does ChatGPT have and what percentage use it for work-related tasks?"


Finally, we can use our `TestSetGenerator` to generate our testset!

In [14]:
testset = generator.generate(testset_size=10, query_distribution=query_distribution)
testset.to_pandas()

Generating personas:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/10 [00:00<?, ?it/s]

,user_input,reference_contexts,reference,synthesizer_name
0,What are Wiggers and how do they relate to Cha...,[Introduction ChatGPT launched in November 202...,"Wiggers, as mentioned in the context, are refe...",single_hop_specifc_query_synthesizer
1,What is the significance of June 2025 in the c...,[Table 1: ChatGPT daily message counts (millio...,"The context reports data ending on June 26, 20...",single_hop_specifc_query_synthesizer
2,What does the variation in ChatGPT usage by oc...,[Variation by Occupation Figure 23 presents va...,The context indicates that ChatGPT usage varie...,single_hop_specifc_query_synthesizer
3,How does ChatGPT provide practical guidance to...,[Conclusion This paper studies the rapid growt...,"According to the context, the most common Chat...",single_hop_specifc_query_synthesizer
4,How do ChatGPT usage topics like Practical Gui...,[<1-hop>\n\nConclusion This paper studies the ...,The context indicates that about 70% of ChatGP...,multi_hop_abstract_query_synthesizer
5,How do user demographics and usage patterns in...,[<1-hop>\n\nIntroduction ChatGPT launched in N...,The data indicates that ChatGPT's usage varies...,multi_hop_abstract_query_synthesizer
6,Based on the data showing that non-work messag...,[<1-hop>\n\nMonth Non-Work (M) (%) Work (M) (%...,The data indicates that non-work messages have...,multi_hop_abstract_query_synthesizer
7,How does the growth in total messages sent to ...,[<1-hop>\n\nIntroduction ChatGPT launched in N...,"By July 2025, ChatGPT users were sending more ...",multi_hop_specific_query_synthesizer
8,How do the details in Appendix B and Appendix ...,[<1-hop>\n\nIntroduction ChatGPT launched in N...,Appendix B provides detailed information on th...,multi_hop_specific_query_synthesizer
9,How does OpenAI's development of ChatGPT contr...,[<1-hop>\n\nIntroduction ChatGPT launched in N...,"OpenAI launched ChatGPT in November 2022, and ...",multi_hop_specific_query_synthesizer


### Abstracted SDG

The above method is the full process - but we can shortcut that using the provided abstractions!

This will generate our knowledge graph under the hood, and will - from there - generate our personas and scenarios to construct our queries.



In [15]:
from ragas.testset import TestsetGenerator

generator = TestsetGenerator(llm=generator_llm, embedding_model=generator_embeddings)
dataset = generator.generate_with_langchain_docs(docs, testset_size=10)

Applying HeadlinesExtractor:   0%|          | 0/21 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/64 [00:00<?, ?it/s]

unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to ap

Applying SummaryExtractor:   0%|          | 0/38 [00:00<?, ?it/s]

Property 'summary' already exists in node 'b993d1'. Skipping!
Property 'summary' already exists in node '9c5fba'. Skipping!
Property 'summary' already exists in node '69a1c1'. Skipping!
Property 'summary' already exists in node '9ab28b'. Skipping!
Property 'summary' already exists in node '345782'. Skipping!
Property 'summary' already exists in node '7cbc50'. Skipping!
Property 'summary' already exists in node '6bee61'. Skipping!
Property 'summary' already exists in node 'da2761'. Skipping!
Property 'summary' already exists in node '326544'. Skipping!
Property 'summary' already exists in node '7de40a'. Skipping!
Property 'summary' already exists in node 'b3edfd'. Skipping!
Property 'summary' already exists in node 'd2cbdc'. Skipping!
Property 'summary' already exists in node 'e8d852'. Skipping!
Property 'summary' already exists in node 'fc1340'. Skipping!
Property 'summary' already exists in node 'ea562d'. Skipping!
Property 'summary' already exists in node '408b79'. Skipping!
Property

...

Property 'summary_embedding' already exists in node '6bee61'. Skipping!
Property 'summary_embedding' already exists in node '69a1c1'. Skipping!
Property 'summary_embedding' already exists in node '345782'. Skipping!
Property 'summary_embedding' already exists in node 'b3edfd'. Skipping!
Property 'summary_embedding' already exists in node 'da2761'. Skipping!
Property 'summary_embedding' already exists in node 'b993d1'. Skipping!
Property 'summary_embedding' already exists in node '7de40a'. Skipping!
Property 'summary_embedding' already exists in node '7cbc50'. Skipping!
Property 'summary_embedding' already exists in node '9ab28b'. Skipping!
Property 'summary_embedding' already exists in node '9c5fba'. Skipping!
Property 'summary_embedding' already exists in node '326544'. Skipping!
Property 'summary_embedding' already exists in node '408b79'. Skipping!
Property 'summary_embedding' already exists in node 'e8d852'. Skipping!
Property 'summary_embedding' already exists in node 'd2cbdc'. Sk

Applying [CosineSimilarityBuilder, OverlapScoreBuilder]:   0%|          | 0/2 [00:00<?, ?it/s]

Generating personas:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/12 [00:00<?, ?it/s]

In [16]:
dataset.to_pandas()

,user_input,reference_contexts,reference,synthesizer_name
0,"What does Korinek and Suh, 2024, say about AI'...",[Introduction ChatGPT launched in November 202...,"The context mentions that Korinek and Suh, 202...",single_hop_specifc_query_synthesizer
1,What does Handa et al. (2025) reveal about the...,[Table 1: ChatGPT daily message counts (millio...,Handa et al. (2025) report that nearly 80% of ...,single_hop_specifc_query_synthesizer
2,SOC2 codes 19 what is that,[Variation by Occupation Figure 23 presents va...,Variation by Occupation Figure 23 presents var...,single_hop_specifc_query_synthesizer
3,What does Seeking Information mean in ChatGPT ...,[Conclusion This paper studies the rapid growt...,Seeking Information is one of the three most c...,single_hop_specifc_query_synthesizer
4,How does the rapid growth and widespread adopt...,[<1-hop>\n\nConclusion This paper studies the ...,The context highlights that since its launch i...,multi_hop_abstract_query_synthesizer
5,How does the rapid growth and adoption of Chat...,[<1-hop>\n\nConclusion This paper studies the ...,"The rapid growth and adoption of ChatGPT, laun...",multi_hop_abstract_query_synthesizer
6,Considering the variation in ChatGPT usage acr...,[<1-hop>\n\nVariation by Occupation Figure 23 ...,The regression analysis of occupation effects ...,multi_hop_abstract_query_synthesizer
7,How does the rapid launch and adoption of Chat...,[<1-hop>\n\nIntroduction ChatGPT launched in N...,The launch of ChatGPT in November 2022 and its...,multi_hop_abstract_query_synthesizer
8,How does the growth in the total number of mes...,[<1-hop>\n\nIntroduction ChatGPT launched in N...,"By July 2025, ChatGPT users were collectively ...",multi_hop_specific_query_synthesizer
9,How does the rapid growth in total messages se...,[<1-hop>\n\nIntroduction ChatGPT launched in N...,"By July 2025, ChatGPT users were sending more ...",multi_hop_specific_query_synthesizer


We'll need to provide our LangSmith API key, and set tracing to "true".

# 🤝 BREAKOUT ROOM #2

## Task 4: LangSmith Dataset

Now we can move on to creating a dataset for LangSmith!

First, we'll need to create a dataset on LangSmith using the `Client`!

We'll name our Dataset to make it easy to work with later.

In [17]:
from langsmith import Client

client = Client()

dataset_name = "Use Case Synthetic Data - AIE8"

langsmith_dataset = client.create_dataset(
    dataset_name=dataset_name,
    description="Synthetic Data for Use Cases"
)

We'll iterate through the RAGAS created dataframe - and add each example to our created dataset!

> NOTE: We need to conform the outputs to the expected format - which in this case is: `question` and `answer`.

In [18]:
for data_row in dataset.to_pandas().iterrows():
  client.create_example(
      inputs={
          "question": data_row[1]["user_input"]
      },
      outputs={
          "answer": data_row[1]["reference"]
      },
      metadata={
          "context": data_row[1]["reference_contexts"]
      },
      dataset_id=langsmith_dataset.id
  )

## Basic RAG Chain

Time for some RAG!


In [19]:
rag_documents = docs

To keep things simple, we'll just use LangChain's recursive character text splitter!


In [20]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 50
)

rag_documents = text_splitter.split_documents(rag_documents)

We'll create our vectorstore using OpenAI's [`text-embedding-3-small`](https://platform.openai.com/docs/guides/embeddings/embedding-models) embedding model.

In [21]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

As usual, we will power our RAG application with Qdrant!

In [22]:
from langchain_community.vectorstores import Qdrant

vectorstore = Qdrant.from_documents(
    documents=rag_documents,
    embedding=embeddings,
    location=":memory:",
    collection_name="Use Case RAG"
)

In [23]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 10})

To get the "A" in RAG, we'll provide a prompt.

In [24]:
from langchain.prompts import ChatPromptTemplate

RAG_PROMPT = """\
Given a provided context and question, you must answer the question based only on context.

If you cannot answer the question based on the context - you must say "I don't know".

Context: {context}
Question: {question}
"""

rag_prompt = ChatPromptTemplate.from_template(RAG_PROMPT)

As is usual: We'll be using `gpt-4.1-mini` for our RAG!

In [25]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4.1-mini")

Finally, we can set-up our RAG LCEL chain!

In [26]:
from operator import itemgetter
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain.schema import StrOutputParser

rag_chain = (
    {"context": itemgetter("question") | retriever, "question": itemgetter("question")}
    | rag_prompt | llm | StrOutputParser()
)

In [27]:
rag_chain.invoke({"question" : "What are people doing with AI these days?"})

'Based on the provided context, people are using AI, particularly generative AI like ChatGPT, in a variety of ways both at work and outside of work. They use AI to perform workplace tasks by augmenting or automating human labor, producing writing, software code, spreadsheets, and other digital products, which distinguishes generative AI from traditional web search engines. AI is also used for seeking information and advice. Additionally, AI serves either as a co-worker producing output or as a co-pilot that gives advice and improves productivity in human problem-solving. Users engage with AI for diverse intents classified as Asking (seeking information or advice), Doing (producing outputs), or Expressing (self-expression, including relationships, personal reflection, games, and role play). Furthermore, generative AI is flexible and used across many economic and occupational tasks.'

## LangSmith Evaluation Set-up

We'll use OpenAI's GPT-4.1 as our evaluation LLM for our base Evaluators.

In [28]:
eval_llm = ChatOpenAI(model="gpt-4.1")

We'll be using a number of evaluators - from LangSmith provided evaluators, to a few custom evaluators!

In [29]:
from langsmith.evaluation import LangChainStringEvaluator, evaluate

qa_evaluator = LangChainStringEvaluator("qa", config={"llm" : eval_llm})

labeled_helpfulness_evaluator = LangChainStringEvaluator(
    "labeled_criteria",
    config={
        "criteria": {
            "helpfulness": (
                "Is this submission helpful to the user,"
                " taking into account the correct reference answer?"
            )
        },
        "llm" : eval_llm
    },
    prepare_data=lambda run, example: {
        "prediction": run.outputs["output"],
        "reference": example.outputs["answer"],
        "input": example.inputs["question"],
    }
)

dopeness_evaluator = LangChainStringEvaluator(
    "criteria",
    config={
        "criteria": {
            "dopeness": "Is this response dope, lit, cool, or is it just a generic response?",
        },
        "llm" : eval_llm
    }
)

#### 🏗️ Activity #2:

Highlight what each evaluator is evaluating.

- `qa_evaluator`: Correctness - is the answer factually accurate?
- `labeled_helpfulness_evaluator`: Helpfulness - is the response useful compared to the reference answer?
- `dopeness_evaluator`: Style - is the response engaging and creative vs. generic?

## LangSmith Evaluation

In [30]:
evaluate(
    rag_chain.invoke,
    data=dataset_name,
    evaluators=[
        qa_evaluator,
        labeled_helpfulness_evaluator,
        dopeness_evaluator
    ],
    metadata={"revision_id": "default_chain_init"},
)

View the evaluation results for experiment: 'loyal-pollution-99' at:
https://smith.langchain.com/o/537402c6-b599-4067-bfda-0f807970bd78/datasets/7fd98bba-35aa-4485-bca8-08661c14b463/compare?selectedSessions=54d9ae9c-48a0-45bc-bcf8-f861f974e011




0it [00:00, ?it/s]

,inputs.question,outputs.output,error,reference.answer,feedback.correctness,feedback.helpfulness,feedback.dopeness,execution_time,example_id,id
0,what happened in july 2025 with chatgpt and ho...,"In July 2025, ChatGPT had more than 700 millio...",None,"In july 2025, chatgpt had over 700 million use...",1,1,0,1.820939,cb52277d-772f-4d3c-9977-8abf0c19a7a8,9b06b28b-7c2a-48ae-bc37-39264b0c0377
1,Considering the rapid growth of ChatGPT usage ...,"By July 2025, ChatGPT had experienced unpreced...",None,"By July 2025, ChatGPT had experienced a signif...",1,1,0,4.246824,9786252d-3c8a-4a7a-86c4-6995d8896336,22f6e931-4fab-4a8a-89d5-640d198108a9
2,How does the rapid growth in total messages se...,The rapid growth in total messages sent to Cha...,None,"By July 2025, ChatGPT users were sending more ...",1,1,0,2.607246,c3783668-a8e1-446b-98b7-f909979bcc79,6d45e709-bc13-488d-b8f3-11e1b672395e
3,How does the growth in the total number of mes...,The total number of messages sent by ChatGPT u...,None,"By July 2025, ChatGPT users were collectively ...",1,1,0,3.729214,c752e798-bef1-4d63-a303-f9f98b5b0314,76a52ff1-de1c-4583-bac9-aec9854f2b21
4,How does the rapid launch and adoption of Chat...,"The rapid launch and adoption of ChatGPT, whic...",None,The launch of ChatGPT in November 2022 and its...,1,1,0,3.787263,2aaa2cae-b4ac-41ce-b632-9711424b71a4,deb4fd19-4167-418e-bf9e-f2eb5adc745c
5,Considering the variation in ChatGPT usage acr...,Based on the provided context:\n\nThe regressi...,None,The regression analysis of occupation effects ...,1,1,0,5.086925,19857db5-e82d-4270-a686-c30bf19ea555,9f6d66bb-0918-4c09-82ac-a5180a8b4f93
6,How does the rapid growth and adoption of Chat...,"The rapid growth and adoption of ChatGPT, whic...",None,"The rapid growth and adoption of ChatGPT, laun...",1,1,0,3.570929,28a94da0-5bd1-4578-b169-0922d2d25ff0,0eed6bf1-bb5f-4f08-af40-4d3f04892b5c
7,How does the rapid growth and widespread adopt...,The rapid growth and widespread adoption of Ch...,None,The context highlights that since its launch i...,1,1,0,3.244166,f63c6bc8-f511-4149-8ca7-be35279e2464,15642d49-e1e2-415a-9ce6-c4015492c5f0
8,What does Seeking Information mean in ChatGPT ...,"Based on the provided context, ""Seeking Inform...",None,Seeking Information is one of the three most c...,1,1,0,1.866417,dee78e00-96fa-4119-9d54-3c49be7e5dad,efd63057-57b0-4deb-9fc6-3c0377f9eaa6
9,SOC2 codes 19 what is that,"Based on the provided context, SOC2 code 19 co...",None,Variation by Occupation Figure 23 presents var...,1,1,0,1.123957,48272af3-7c20-47ae-878f-14ccfa091e3a,04b87976-2798-4da3-959e-3a928c4b8a55


## Dope-ifying Our Application

We'll be making a few changes to our RAG chain to increase its performance on our SDG evaluation test dataset!

- Include a "dope" prompt augmentation
- Use larger chunks
- Improve the retriever model to: `text-embedding-3-large`

Let's see how this changes our evaluation!

In [31]:
DOPENESS_RAG_PROMPT = """\
Given a provided context and question, you must answer the question based only on context.

If you cannot answer the question based on the context - you must say "I don't know".

Make your answer rad, ensure high levels of dopeness. Do not be generic, or give generic responses.

Context: {context}
Question: {question}
"""

dopeness_rag_prompt = ChatPromptTemplate.from_template(DOPENESS_RAG_PROMPT)

In [32]:
rag_documents = docs

In [33]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 50
)

rag_documents = text_splitter.split_documents(rag_documents)

#### ❓Question #2:

Why would modifying our chunk size modify the performance of our application?

##### ✅ Answer:

Larger chunks (1000): More context per chunk means retrieved documents contain more complete information, potentially improving answer quality but may include irrelevant details.

In [34]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

#### ❓Question #3:

Why would modifying our embedding model modify the performance of our application?

##### ✅ Answer:

Higher dimensional embeddings (3072 dimensions for text-embedding-3-large) capture more nuanced semantic relationships, leading to better retrieval of conceptually similar content

In [35]:
vectorstore = Qdrant.from_documents(
    documents=rag_documents,
    embedding=embeddings,
    location=":memory:",
    collection_name="Use Case RAG Docs"
)

In [36]:
retriever = vectorstore.as_retriever()

Setting up our new and improved DOPE RAG CHAIN.

In [37]:
dopeness_rag_chain = (
    {"context": itemgetter("question") | retriever, "question": itemgetter("question")}
    | dopeness_rag_prompt | llm | StrOutputParser()
)

Let's test it on the same output that we saw before.

In [38]:
dopeness_rag_chain.invoke({"question" : "How are people using AI to make money?"})

'Alright, let’s crank this up to eleven on the dopeness scale: People are not just using AI to grind out tasks; they’re tapping into ChatGPT and generative AI as their secret weapon for making bank by leveling up their decision-making game. According to the mind-blowing insights from Collis and Brynjolfsson (2025), folks are getting massive value by using AI as an advisor or research assistant — think of it as having a hyper-smart sidekick that boosts productivity especially in knowledge-heavy gigs where your choices basically write your paycheck.\n\nSo instead of just automating chores, AI is flexing as a dope decision-support system that magnifies the quality of your work output, translating that sharp advice and info into real cash. This means people cash in by augmenting their brainpower with AI’s know-how, unlocking smarter, faster, and better moves that rake in the stacks. Ain’t no mere task robot here—this is your growth hacker and research guru rolled into one, fueling a killer

Finally, we can evaluate the new chain on the same test set!

In [39]:
evaluate(
    dopeness_rag_chain.invoke,
    data=dataset_name,
    evaluators=[
        qa_evaluator,
        labeled_helpfulness_evaluator,
        dopeness_evaluator
    ],
    metadata={"revision_id": "dopeness_rag_chain"},
)

View the evaluation results for experiment: 'dear-turn-96' at:
https://smith.langchain.com/o/537402c6-b599-4067-bfda-0f807970bd78/datasets/7fd98bba-35aa-4485-bca8-08661c14b463/compare?selectedSessions=dcaf762b-f957-4f9b-b006-716c4299dcff




0it [00:00, ?it/s]

,inputs.question,outputs.output,error,reference.answer,feedback.correctness,feedback.helpfulness,feedback.dopeness,execution_time,example_id,id
0,what happened in july 2025 with chatgpt and ho...,"Yo, July 2025 was straight fire for ChatGPT! T...",None,"In july 2025, chatgpt had over 700 million use...",0,0,1,2.053314,cb52277d-772f-4d3c-9977-8abf0c19a7a8,b72e532e-91bc-4fe8-8fab-a915238236f6
1,Considering the rapid growth of ChatGPT usage ...,"Alright, strap in for a turbo-charged dive int...",None,"By July 2025, ChatGPT had experienced a signif...",1,1,1,6.563933,9786252d-3c8a-4a7a-86c4-6995d8896336,7f73efba-6d2f-4907-a5f2-eef7cc67a93e
2,How does the rapid growth in total messages se...,"Alright, here’s the lowdown – the message tsun...",None,"By July 2025, ChatGPT users were sending more ...",1,1,1,3.329661,c3783668-a8e1-446b-98b7-f909979bcc79,165ba6c5-16e1-4056-acd7-de69b7e83c7d
3,How does the growth in the total number of mes...,"Yo, strap in for this data drop straight from ...",None,"By July 2025, ChatGPT users were collectively ...",1,0,1,3.671982,c752e798-bef1-4d63-a303-f9f98b5b0314,397dba1f-e9d1-408d-a2a5-ca6651a7196a
4,How does the rapid launch and adoption of Chat...,"Alright, let’s crank up the dopeness and decod...",None,The launch of ChatGPT in November 2022 and its...,1,1,1,6.757109,2aaa2cae-b4ac-41ce-b632-9711424b71a4,dc313f5f-3b42-4afc-9d35-61a7426ce888
5,Considering the variation in ChatGPT usage acr...,"Alright, let’s unwrap this AI mystery like a d...",None,The regression analysis of occupation effects ...,1,1,1,11.333021,19857db5-e82d-4270-a686-c30bf19ea555,e71c0330-ab8c-41de-99a3-06afdad01ec7
6,How does the rapid growth and adoption of Chat...,"Yo, let's unpack this AI saga with some seriou...",None,"The rapid growth and adoption of ChatGPT, laun...",1,1,1,4.543381,28a94da0-5bd1-4578-b169-0922d2d25ff0,fad1c034-f4d2-4038-81d4-c9c46ae50974
7,How does the rapid growth and widespread adopt...,"Alright, buckle up — here’s why the rocket-spe...",None,The context highlights that since its launch i...,1,1,1,6.608257,f63c6bc8-f511-4149-8ca7-be35279e2464,0843eb51-ca7a-4508-87e4-b73f0f46dc29
8,What does Seeking Information mean in ChatGPT ...,"Alright, buckle up for some next-level AI wisd...",None,Seeking Information is one of the three most c...,0,0,1,2.439685,dee78e00-96fa-4119-9d54-3c49be7e5dad,cc8f0d01-e57a-46ab-9f32-d8f55decaad9
9,SOC2 codes 19 what is that,Boom! SOC2 code 19 is all about **Science**. I...,None,Variation by Occupation Figure 23 presents var...,1,1,1,1.849499,48272af3-7c20-47ae-878f-14ccfa091e3a,5b8fce25-9d3a-4e07-858b-9dee23ff7997


#### 🏗️ Activity #3:

Provide a screenshot of the difference between the two chains, and explain why you believe certain metrics changed in certain ways.

![alt text](image.png)

On the image above "loyal-polltion-99" is default and "dear-turn-96" is dopped chain.

We can see that only dopeness is higher on a cost of correctness and helpfulness dropping.

We can explain certain behaviours in trend with following:
1. Correctness - "Dopeness" prompt might add fluff that obscures facts
2. Helpfulness - Overly stylized answers may be seen as less helpful for factual questions
3. Dopeness - Explicit instruction to ensure high level of dopeness did exactly that